# TaaS Hypergraph Discovery Projection — Colab validation

**Feature branch:** `feature/hypergraph-discovery-projection`  
**Pinned implementation commit:** `b7c8e53a7fd9aa01a27bf85464c0852954776702`

This notebook validates the read-only projection of the actual TaaS `Hypergraph` and `ReachResult` core objects. It does not implement graph reasoning, certify a toolkit, authorize a tool, or execute MCP calls. All examples are synthetic test fixtures.

In [ ]:
import pathlib, subprocess, sys
REPO_URL = 'https://github.com/msellamiTN/taas-provenance-discovery.git'
REVISION = 'b7c8e53a7fd9aa01a27bf85464c0852954776702'
WORKDIR = pathlib.Path('/content/taas-provenance-discovery')
if not WORKDIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
subprocess.run(['git', 'fetch', '--all'], cwd=WORKDIR, check=True)
subprocess.run(['git', 'checkout', '--detach', REVISION], cwd=WORKDIR, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=WORKDIR, text=True).strip()
assert actual == REVISION, (actual, REVISION)
SRC = WORKDIR / 'src'
assert (SRC / 'taas' / '__init__.py').is_file()
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
print('Pinned revision:', actual)
print('Package source:', SRC / 'taas')


In [ ]:
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[test]'], cwd=WORKDIR, text=True, capture_output=True)
print(install.stdout[-3000:])
if install.returncode:
    print(install.stderr[-6000:])
    raise RuntimeError('TaaS dependency installation failed')


In [ ]:
from taas.contracts import make_contract
from taas.hypergraph import Derivation, Hypergraph, ModelEdge, ReachResult
from taas.hypergraph_projection import project_hypergraph, PROJECTION_SCHEMA
from taas.types import parse_type

graph = Hypergraph()
contract = make_contract('orders::list_orders', {'customer_id': 'identifier', 'limit': 'number'}, 'record:order[]', ['read(orders)'], optional=('limit',), description='List orders', source_label='internal')
graph.add(contract)
reach = ReachResult(
    types={parse_type('identifier'), parse_type('record:order[]')},
    derivations={
        'identifier': Derivation('prompt', 'identifier', (), 'prompt'),
        'record:order[]': Derivation(contract.tool, 'record:order[]', ('identifier',), 'tool', 0.95),
    },
    bridges=[], labels={'identifier': 'public', 'record:order[]': 'internal'},
    fired={contract.tool}, tool_types={'record:order[]': contract.tool},
)
projection = project_hypergraph(graph, reach)
assert projection['schema'] == PROJECTION_SCHEMA
assert projection['summary']['registered_tool_count'] == 1
edge = next(x for x in projection['hyperedges'] if x['kind'] == 'tool')
assert edge['required_inputs'] == ['identifier']
assert edge['head'] == 'record:order[]'
assert edge['fired_in_reachability'] is True
nodes = {x['semantic_type']: x for x in projection['vertices']}
assert nodes['record:order[]']['reachable'] is True
assert nodes['record:order[]']['information_label'] == 'internal'
print('PASS: actual core objects projected')
print('Projection ID:', projection['projection_id'])
print('Summary:', projection['summary'])


In [ ]:
model = ModelEdge(cls='summarise', tail=(parse_type('record:order[]'),), head=parse_type('text'), rho=0.82, goal='g-summary')
reach_with_model = ReachResult(
    types={parse_type('record:order[]'), parse_type('text')},
    derivations={'text': Derivation(model.id, 'text', ('record:order[]',), 'model', 0.82)},
    bridges=[Derivation('LLM::extract', 'identifier', ('text',), 'bridge', 0.75)],
    labels={'record:order[]': 'internal', 'text': 'internal'}, fired={model.id},
)
model_projection = project_hypergraph(graph, reach_with_model, selected_tools=[], model_edges=[model])
assert any(x['kind'] == 'model' and x['goal_id'] == 'g-summary' for x in model_projection['hyperedges'])
assert len(model_projection['bridges']) == 1
print('PASS: model edge and recorded bridge projected')


In [ ]:
tests = subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_hypergraph_projection.py'], cwd=WORKDIR, text=True, capture_output=True)
print(tests.stdout)
if tests.returncode:
    print(tests.stderr[-8000:])
    raise RuntimeError('Hypergraph projection tests failed')
print('PASS: focused pytest suite')


## Validation boundary

A successful run establishes that the projection serializes the tested core graph, reachability, derivations, information labels, model edges and bridge records at the pinned revision. It does **not** establish API integration, frontend correctness, production authorization, complete policy semantics, or governed execution. Record the actual Colab output and runtime timestamp before treating this feature as runtime-validated.